In [ ]:
import os
import sqlite3

import geopandas as gpd
import osmium
import numpy as np
import pandas as pd
import sys
sys.path.append('../')

CRS = "EPSG:31370"  # Belgian Lambert 72

In [ ]:
from constants import brussels_muni_codes_str, brussels_muni_codes

In [ ]:
RNG = np.random.default_rng(42)

# 1 ASSIGN WORK STATISTICAL SECTOR

In [ ]:
def calculate_commute_probabilities(db_path):
    """
    Calculate the probability that a person from statistical sector i commutes to sector j.
    It only returns statistical sectors i from BCR and excludes ZZZZ sectors and FOR (abroad).

    Args:
        db_path: Path to the SQLite database file

    Returns:
        DataFrame with home_sector, work_sector, and probability columns
    """
    # Connect to the database
    conn = sqlite3.connect(db_path)

    # Check if the database file exists
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"The database file at {db_path} does not exist.")

    # Read the data
    query = """
    SELECT 
        CD_SECTOR_RESIDENCE AS home_statistical_sector,
        CASE WHEN CD_SECTOR_WORK = 'BE10Z_ZZZZZZZZZ' THEN '_04000'            -- extra _ so that the split works correctly and keeps 04000_ZZZZZ
            ELSE CD_SECTOR_WORK END AS work_statistical_sector, 
        CAST(OBS_VALUE AS INTEGER) AS num_people
    FROM 'VU_GEO_LPW_MATRIX'
    WHERE CD_RGN_RESIDENCE = '04000'
    AND CD_SECTOR_RESIDENCE NOT LIKE '%ZZZZ'
    AND CD_SECTOR_WORK NOT LIKE 'FOR'
    AND (CD_SECTOR_WORK = 'BE10Z_ZZZZZZZZZ' OR CD_SECTOR_WORK NOT LIKE '%ZZZZZZ')   -- only include BCR for the ZZZZZZ sectors
    """
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    # Extract the part after underscore for sector codes
    df["home_statistical_sector"] = df["home_statistical_sector"].str.split("_").str[1]
    df["work_statistical_sector"] = df["work_statistical_sector"].str.split("_").str[1]


    # Calculate total people from each home sector
    total_by_home = (
        df.groupby("home_statistical_sector")["num_people"].sum().reset_index()
    )
    total_by_home.columns = ["home_statistical_sector", "total_home"]

    # Merge and calculate probabilities
    df = df.merge(total_by_home, on="home_statistical_sector")
    df["probability"] = df["num_people"] / df["total_home"]

    # Select final columns
    result = df[["home_statistical_sector", "work_statistical_sector", "probability", "num_people"]]

    return result

In [ ]:
db_path = "input_data/census_matrix_home_work_ss_2011.sqlite"

probabilities = calculate_commute_probabilities(db_path)
print(probabilities.head())

# Save to CSV
probabilities.to_csv("output/commute_probabilities.csv", index=False)

## Re-distribute the same-sector flows

In [ ]:
def redistribute_same_sector_flows(result_df, target_diagonal_share=0.04):
    """
    Redistribute same-sector flows so they account for ~target_diagonal_share
    of each home sector's total, spreading the excess to other destinations
    proportionally.

    Args:
        result_df: DataFrame from calculate_commute_probabilities(), with
                   columns home_statistical_sector, work_statistical_sector,
                   probability, num_people
        target_diagonal_share: desired share for same-sector flows (default 4%)

    Returns:
        DataFrame with adjusted probability and num_people columns
    """
    df = result_df.copy()

    adjusted_rows = []

    for home_sector, group in df.groupby("home_statistical_sector"):
        total = group["num_people"].sum()

        # Split diagonal and off-diagonal
        is_diagonal = group["work_statistical_sector"] == home_sector
        diag_row = group[is_diagonal].copy()
        off_diag = group[~is_diagonal].copy()

        current_diag = diag_row["num_people"].values[0] if len(diag_row) > 0 else 0
        target_diag = target_diagonal_share * total

        if current_diag <= target_diag or len(off_diag) == 0:
            # Nothing to redistribute
            adjusted_rows.append(group)
            continue

        # Excess to redistribute
        excess = current_diag - target_diag

        # Distribute excess proportionally to off-diagonal destinations
        off_diag_total = off_diag["num_people"].sum()
        off_diag = off_diag.copy()
        off_diag["num_people"] = off_diag["num_people"] + excess * (off_diag["num_people"] / off_diag_total)

        # Update diagonal
        diag_row = diag_row.copy()
        diag_row["num_people"] = target_diag

        adjusted_rows.append(pd.concat([diag_row, off_diag]))

    df_adjusted = pd.concat(adjusted_rows, ignore_index=True)

    # Recompute probabilities from adjusted num_people
    total_by_home = (
        df_adjusted.groupby("home_statistical_sector")["num_people"]
        .sum()
        .reset_index()
        .rename(columns={"num_people": "total_home"})
    )
    df_adjusted = df_adjusted.merge(total_by_home, on="home_statistical_sector")
    df_adjusted["probability"] = df_adjusted["num_people"] / df_adjusted["total_home"]
    df_adjusted = df_adjusted.drop(columns="total_home")
    df_adjusted['num_people'] = df_adjusted['num_people'].round().astype(int)

    return df_adjusted

In [ ]:
probabilities = redistribute_same_sector_flows(probabilities, 0.04)

# Save to CSV
probabilities.to_csv("output/commute_probabilities.csv", index=False)

In [ ]:
# Check new share of same-sector flows
probabilities[probabilities.home_statistical_sector == probabilities.work_statistical_sector].groupby("home_statistical_sector")["probability"].sum().sort_values(ascending=False).head(10)

## Apply IPF to get 2021 totals

In [ ]:
def apply_ipf(probabilities_2011, df_2021_long, max_iter=500, tol=1e-6):
    """
    Adjust 2011 sector-level OD flows using IPF to match 2021 municipality-level marginals.
    
    Args:
        probabilities_2011: DataFrame with home_statistical_sector, work_statistical_sector, 
                            probability, num_people
        df_2021_long: DataFrame with home_muni_code, work_muni_code, num_people
        max_iter: Maximum number of IPF iterations
        tol: Convergence tolerance
    
    Returns:
        DataFrame with adjusted probabilities at sector level
    """
    df = probabilities_2011.copy()

    # Extract municipality codes from sector codes (first 5 digits)
    df["home_muni"] = df["home_statistical_sector"].str[:5].astype(int)
    df["work_muni"] = df["work_statistical_sector"].str[:5].astype(int)

    # Work with flows rather than probabilities for IPF
    df["flow"] = df["num_people"].astype(float)

    # Build 2021 municipality-level marginals
    # Origin marginals: total commuters leaving each home municipality
    origin_marginals = (
        df_2021_long.groupby("home_muni_code")["num_people"]
        .sum()
        .to_dict()
    )
    # Destination marginals: total commuters arriving at each work municipality
    dest_marginals = (
        df_2021_long.groupby("work_muni_code")["num_people"]
        .sum()
        .to_dict()
    )

    # IPF iterations
    for iteration in range(max_iter):
        df_prev = df["flow"].copy()

        # Step 1: Scale by origin municipality marginals
        origin_current = df.groupby("home_muni")["flow"].sum()
        for muni, target in origin_marginals.items():
            if muni not in origin_current.index or origin_current[muni] == 0:
                continue
            scale = target / origin_current[muni]
            df.loc[df["home_muni"] == muni, "flow"] *= scale

        # Step 2: Scale by destination municipality marginals
        dest_current = df.groupby("work_muni")["flow"].sum()
        for muni, target in dest_marginals.items():
            if muni not in dest_current.index or dest_current[muni] == 0:
                continue
            scale = target / dest_current[muni]
            df.loc[df["work_muni"] == muni, "flow"] *= scale

        # Check convergence
        max_change = (df["flow"] - df_prev).abs().max()
        if iteration % 20 == 0:
            print(f"Iteration {iteration + 1}: max change = {max_change:.6f}")
        if max_change < tol:
            print(f"Converged after {iteration + 1} iterations")
            break

    # Recalculate probabilities from adjusted flows
    total_by_home = df.groupby("home_statistical_sector")["flow"].sum()
    df["probability_adjusted"] = df.apply(
        lambda row: row["flow"] / total_by_home[row["home_statistical_sector"]]
        if total_by_home[row["home_statistical_sector"]] > 0 else 0,
        axis=1
    )

    return df[["home_statistical_sector", "work_statistical_sector", 
               "probability", "probability_adjusted", "flow",
               "home_muni", "work_muni"]]

In [ ]:
# Run IPF
df_raw_2021 = pd.read_csv('input_data/working_population_home_work_municipality_2021_BCR_all.csv')

# Reshape df_raw to long format for analysis
df_2021 = df_raw_2021.copy()

# Set the index to 'Code INS' and extract municipality codes
df_2021 = df_2021.set_index('Code INS')
df_2021 = df_2021.drop('Lieu de résidence', axis=1)

# Change to long format
df_2021_long = df_2021.reset_index().melt(
    id_vars='Code INS',
    var_name='work_municipality_code',
    value_name='count'
)

# Rename columns for consistency
df_2021_long = df_2021_long.rename(columns={
    'Code INS': 'home_muni_code',
    'work_municipality_code': 'work_muni_code',
    'count': 'num_people'
})

# Convert to numeric
df_2021_long['home_muni_code'] = pd.to_numeric(df_2021_long['home_muni_code'], errors='coerce')
df_2021_long['work_muni_code'] = pd.to_numeric(df_2021_long['work_muni_code'], errors='coerce')

probabilities_adjusted = apply_ipf(probabilities, df_2021_long)

# Save result
probabilities_adjusted.to_csv("output/commute_probabilities_adjusted.csv", index=False)

## Assign work sector

In [ ]:
def assign_work_sectors(synthetic_pop, commute_probabilities):
    """
    Assign a work sector to each agent in the synthetic population based on commute probabilities.

    Args:
        synthetic_pop: DataFrame with synthetic population (must include 'sector' column for home sector)
        commute_probabilities: DataFrame with home_statistical_sector, work_statistical_sector, and probability columns
    Returns:
        DataFrame with an additional 'work_sector' column assigned to each agent
    """

    # Create a dictionary with one entry for each origin sector, and the available work sectors and probabilities
    od_prob = {}
    for home_sector, group in commute_probabilities.groupby("home_statistical_sector"):
        od_prob[home_sector] = {
            "work_sectors": group["work_statistical_sector"].values,
            "probs": group["probability_adjusted"].values,
        }

    # Create fallback in case home_statistical_sector has now entry in the commuting probabilities
    muni_probs = {}
    for home_muni, group in commute_probabilities.groupby("home_muni"):
        muni_probs[home_muni] = {
            "work_sectors": group["work_statistical_sector"].values,
            "probs": group["flow"].values / group["flow"].values.sum(),
        }

    global_probs = (
        commute_probabilities
        .groupby("work_statistical_sector")["flow"]
        .sum()
    )

    global_probs = global_probs / global_probs.sum()

    def sample_work_sector(home_sector):
        if home_sector in od_prob:
            work_sector = RNG.choice(
                od_prob[home_sector]["work_sectors"], p=od_prob[home_sector]["probs"]
            )

        elif home_sector[:5] in muni_probs:
            work_sector = RNG.choice(
                muni_probs[home_sector[:5]]["work_sectors"], p=muni_probs[home_sector[:5]]["probs"]
            )
            print(f'Using muni probs for stat sector {home_sector}')

        else:
            work_sector = RNG.choice(global_probs.index, p=global_probs.values)

        # Fallback
        if work_sector is None:
            work_sector = RNG.choice(global_probs.index, p=global_probs.values)
            print(f'Using global probs for stat sector {home_sector}')
        
        return work_sector


    synthetic_pop["work_sector"] = synthetic_pop["assigned_sector"].apply(sample_work_sector)

    return synthetic_pop

In [ ]:
# Get all working people
workers = pd.read_csv(
    "../2_home_assignment/output/employed_population_with_home_address.csv"
)

In [ ]:
# Assign work sectors based on probabilities (with updated function - fresh run)
workers_with_work_sector = assign_work_sectors(workers.copy(), probabilities_adjusted)

# Save
workers_with_work_sector.to_csv("output/workers_with_work_sector.csv", index=False)

In [ ]:
# Check if each worker has a work_sector assigned
print(f"Total workers: {len(workers)}")
print(f"Total workers with work sector: {len(workers_with_work_sector)}")
print(f"Workers with work_sector assigned: {workers_with_work_sector['work_sector'].notna().sum()}")
print(f"Workers missing work_sector: {workers_with_work_sector['work_sector'].isna().sum()}")
print(f"Workers with ZZZZ work_sector: {workers_with_work_sector['work_sector'].str.endswith('ZZZZ').sum()}")

In [ ]:
def fix_zzz_work_sector(synthetic_pop, commute_probabilities):
    """
    Assign a work sector to each agent in the synthetic population based on commute probabilities.

    Args:
        synthetic_pop: DataFrame with synthetic population (must include 'sector' column for home sector)
        commute_probabilities: DataFrame with home_statistical_sector, work_statistical_sector, and probability columns
    Returns:
        DataFrame with an additional 'work_sector' column assigned to each agent
    """

    # Remove ZZZZ work sectors from the probs
    commute_probabilities = commute_probabilities[~commute_probabilities['work_statistical_sector'].str.endswith('ZZZZ')].copy()

    # Remove 04000 from the probs
    commute_probabilities = commute_probabilities[~commute_probabilities['work_statistical_sector'].str.endswith('04000')].copy()

    # Create a dictionary with one entry for each origin muni, and the available work sectors and probabilities
    muni_probs = {}
    for work_muni, group in commute_probabilities.groupby("work_muni"):
        muni_probs[work_muni] = {
            "work_sectors": group["work_statistical_sector"].values,
            "probs": group["flow"].values / group["flow"].values.sum(),
        }

    bcr_probs = {
        "work_sectors": commute_probabilities[commute_probabilities['work_muni'].astype(int).isin(brussels_muni_codes)]['work_statistical_sector'].values,
        "probs": commute_probabilities[commute_probabilities['work_muni'].astype(int).isin(brussels_muni_codes)]['flow'].values / commute_probabilities[commute_probabilities['work_muni'].astype(int).isin(brussels_muni_codes)]['flow'].values.sum(),
    }

    global_probs = (
        commute_probabilities
        .groupby("work_statistical_sector")["flow"]
        .sum()
    )

    global_probs = global_probs / global_probs.sum()

    def sample_new_work_sector(work_sector):
        if work_sector == '04000':
            work_sector = RNG.choice(bcr_probs["work_sectors"], p=bcr_probs["probs"])
        elif work_sector.endswith('ZZZZ') and int(work_sector[:5]) in muni_probs:
            work_sector = RNG.choice(
                muni_probs[int(work_sector[:5])]["work_sectors"], p=muni_probs[int(work_sector[:5])]["probs"]
            )
        elif work_sector.endswith('ZZZZ') and int(work_sector[:5]) not in muni_probs:
            work_sector = RNG.choice(global_probs.index, p=global_probs.values)
        
        return work_sector


    synthetic_pop["work_sector"] = synthetic_pop["work_sector"].apply(sample_new_work_sector)

    return synthetic_pop

In [ ]:
# Assign new work sectors for ZZZZ work sector
workers_with_work_sector = fix_zzz_work_sector(workers_with_work_sector.copy(), probabilities_adjusted)

# Save
workers_with_work_sector.to_csv("output/workers_with_work_sector.csv", index=False)

In [ ]:
# Check if each worker has a work_sector assigned
print(f"Total workers: {len(workers)}")
print(f"Total workers with work sector: {len(workers_with_work_sector)}")
print(f"Workers with work_sector assigned: {workers_with_work_sector['work_sector'].notna().sum()}")
print(f"Workers missing work_sector: {workers_with_work_sector['work_sector'].isna().sum()}")
print(f"Workers with ZZZZ work_sector: {workers_with_work_sector['work_sector'].str.endswith('ZZZZ').sum()}")
print(f"Workers with 04000 work_sector: {workers_with_work_sector['work_sector'].str.endswith('04000').sum()}")

# 2 ASSIGN WORK ADDRESS

In [ ]:
# Get all nodes on highways in the OSM data
class WayNodeHandler(osmium.SimpleHandler):
    def __init__(self):
        super().__init__()
        self.highway_node_ids = set()
    
    def way(self, w):
        if 'highway' in w.tags:
            for n in w.nodes:
                self.highway_node_ids.add(n.ref)

class NodeLocationHandler(osmium.SimpleHandler):
    def __init__(self, node_ids):
        super().__init__()
        self.node_ids = node_ids
        self.nodes = []
    
    def node(self, n):
        if n.id in self.node_ids:
            self.nodes.append({
                'node_id': n.id,
                'lat': n.location.lat,
                'lon': n.location.lon
            })

# Collect node IDs from highway ways
way_handler = WayNodeHandler()
way_handler.apply_file('input_data/belgium.osm.pbf')
print(f"Highway node IDs collected: {len(way_handler.highway_node_ids)}")

# Extract coordinates for those nodes
node_handler = NodeLocationHandler(way_handler.highway_node_ids)
node_handler.apply_file('input_data/belgium.osm.pbf', locations=True)

df_nodes = pd.DataFrame(node_handler.nodes)
print(f"Total highway nodes with coordinates: {len(df_nodes)}")

In [ ]:
# Convert to GeoDataFrame in WGS84 then reproject to Lambert
gdf_nodes = gpd.GeoDataFrame(
    df_nodes,
    geometry=gpd.points_from_xy(df_nodes['lon'], df_nodes['lat']),
    crs='EPSG:4326'
).to_crs(CRS)

# Load municipality boundaries from statistical sectors shapefile
gdf_sectors = gpd.read_file('input_data/statistical_sectors.shp').to_crs(CRS)
gdf_sectors['CNIS5_2020'] = gdf_sectors['CNIS5_2020'].astype(str).str.zfill(5)

# Dissolve sectors to municipality level
gdf_municipalities = gdf_sectors.dissolve(by='CNIS5_2020').reset_index()
print(f"Total municipalities: {len(gdf_municipalities)}")

# Spatial join nodes to municipalities
gdf_nodes_joined = gpd.sjoin(
    gdf_nodes,
    gdf_municipalities[['CNIS5_2020', 'geometry']],
    how='inner',
    predicate='within'
)

print(f"Nodes with municipality assignment: {len(gdf_nodes_joined)}")
print(f"Municipalities with at least one node: {gdf_nodes_joined['CNIS5_2020'].nunique()}")

# Build lookup dictionary
nodes_by_municipality = gdf_nodes_joined.groupby('CNIS5_2020').apply(
    lambda x: x['geometry'].values.tolist(),
    include_groups=False
).to_dict()

print("\nSample municipality node counts:")
for muni, nodes in list(nodes_by_municipality.items())[:5]:
    print(f"  {muni}: {len(nodes)} nodes")

In [ ]:
def get_work_addresses(muni_nodes):
    """
    Load addresses and filter for those suitable for office/work use.
    For Brussels sectors: use actual addresses from work zones (land use).
    For non-Brussels sectors: use either a node on the network for the municipality
    or the centroid of the statistical sector.

    Args:
        muni_nodes: nodes on the network to be used for the addresses outside of Brussels

    Returns:
        Dictionary mapping sector to list of work addresses (geometries)
    """
    # Load addresses file
    addresses = gpd.read_file("input_data/urbis_addresses.shp")

    # Load land use plan
    land_use = gpd.read_file("input_data/pras.shp")

    # Filter for office/work land use types
    work_land_use = land_use[
        land_use["AFFECTATIO"].isin(
            [
                "AfZFMixite",
                "AfZIndustrie",
                "AfZTransPort",
                "AfZAdmin",
                "AfZEquipt",
                "AfZMixite",
                "AfZEntrMilUrb",
                "AfZChFer",
                "AfZir",
            ]
        )
    ]

    # Ensure both use the same CRS format (EPSG:31370 = Belgian Lambert 72)
    addresses = addresses.to_crs(CRS)
    work_land_use = work_land_use.to_crs(CRS)

    # Merge addresses with land use data and filter for work zones
    sector_addresses = {}
    addresses_with_landuse = addresses.sjoin(work_land_use, how="inner")

    for sector, group in addresses_with_landuse.groupby("STATNISCOD"):
        sector_addresses[sector] = group.geometry.tolist()

    # Load statistical sectors for all sectors (including non-Brussels)
    statistical_sectors = gpd.read_file(
        "input_data/statistical_sectors.shp"
    )
    statistical_sectors = statistical_sectors.to_crs(CRS)

    # For sectors without work addresses assign centroid or node from the network
    for sector in statistical_sectors["CS01012020"].unique():
        if sector not in sector_addresses:
            # For Brussels stat sectors use the centroid
            if sector[:5] in brussels_muni_codes_str or sector[:5] not in muni_nodes:
                sector_geom = statistical_sectors[
                    statistical_sectors["CS01012020"] == sector
                ]
                if not sector_geom.empty:
                    centroid = sector_geom.geometry.centroid.values[0]
                    sector_addresses[sector] = [centroid]
            else:
                sector_addresses[sector] = muni_nodes[sector[:5]]

    return sector_addresses

In [ ]:
# Get all possible work addresses (offices/industrial zones etc) for Brussels
# + nodes on the network or centroids of statistical sectors outside of Brussels
work_addresses = get_work_addresses(nodes_by_municipality)

In [ ]:
def assign_work_address(synthetic_pop, sector_addresses):
    """
    Assign a random work address to each agent based on their assigned work sector.

    Args:
        synthetic_pop: DataFrame with synthetic population (must include 'work_sector' column)
        sector_addresses: Dictionary mapping work_sector to a list of possible addresses (geometries)

    Returns:
        DataFrame with an additional 'work_geometry' column assigned to each agent
    """

    def assign_random_work_address(work_sector):
        if work_sector in sector_addresses:
            return RNG.choice(sector_addresses[work_sector])
        else:
            return None  # If no addresses for this sector
    
    synthetic_pop["work_geometry"] = synthetic_pop["work_sector"].apply(
        assign_random_work_address
    )

    # Convert to GeoDataFrame
    pop_gdf = gpd.GeoDataFrame(synthetic_pop, geometry="work_geometry", crs=CRS)

    # Extract X/Y coordinates if needed
    pop_gdf["work_x"] = pop_gdf.geometry.x
    pop_gdf["work_y"] = pop_gdf.geometry.y

    # Save to file
    pop_gdf.to_file("output/workers_with_work_addresses.shp")
    pop_gdf[
        [
            "sex",
            "age",
            "education",
            "industry",
            "professional_status",
            "sector",
            "assigned_sector",
            "home_x",
            "home_y",
            "work_sector",
            "work_x",
            "work_y",
        ]
    ].to_csv("output/workers_with_work_addresses.csv", index=False)

    return pop_gdf

In [ ]:
# Assign work addresses to workers
# Takes around 10 minutes to run
workers_with_work_addresses = assign_work_address(
    workers_with_work_sector, work_addresses
)

In [ ]:
# Check how many workers have missing work sectors or addresses
df = workers_with_work_addresses

total_workers = len(df)
no_work_sector = df["work_sector"].isna().sum()
no_assigned_work_sector = df["work_sector"].isna().sum()

if {"work_x", "work_y"}.issubset(df.columns):
    no_work_coordinates = (df["work_x"].isna() | df["work_y"].isna()).sum()
elif "work_geometry" in df.columns:
    no_work_coordinates = df["work_geometry"].isna().sum()
else:
    raise KeyError("No work coordinate columns found (expected work_x/work_y or work_geometry).")

municipality_codes = [21001, 21002, 21003, 21004, 21005, 21006, 21007, 21008, 21009, 21010, 21011, 21012, 21013, 21014, 21015, 21016, 21017, 21018, 21019]

print(f"Total workers: {total_workers:,}")
print(f"Workers working in Brussels: {len(df[df['work_sector'].str.slice(0, 5).astype(int).isin(municipality_codes)])} ")
print(f"No work_sector: {no_work_sector:,} ({no_work_sector / total_workers:.2%})")
print(f"No assigned_work_sector: {no_assigned_work_sector:,} ({no_assigned_work_sector / total_workers:.2%})")
print(f"No work coordinates: {no_work_coordinates:,} ({no_work_coordinates / total_workers:.2%})")

In [ ]:
# Check agents assigned to municipalities that do not have any nodes in the network
missing_munis = set(gdf_municipalities['CNIS5_2020']) - set(gdf_nodes_joined['CNIS5_2020'])
agents_in_missing = df[df['work_sector'].str[:5].isin(missing_munis)]
print(f"Agents in unreachable municipalities: {len(agents_in_missing)} ({len(agents_in_missing)/len(df):.2%})")

In [ ]:
# Drop workers without an assigned work address + with work address not on the network
# There were only a very small number of workers without work sectors or coordinates, so we can safely drop them for the next steps
df = workers_with_work_addresses.copy()

df = df[df["work_x"].notna() & df["work_y"].notna()].copy()
df = df[~df['work_sector'].str[:5].isin(missing_munis)]

df.to_csv("output/workers_with_work_addresses.csv", index=False)